In [2]:
import requests
from bs4 import BeautifulSoup
import time

In [3]:
apartments = []
seen_links = []

for page in range(1, 300):
    page_url = f"https://krisha.kz/prodazha/kvartiry/astana/?page={page}"
    response = requests.get(page_url)

    print("Page:", page, "Status:", response.status_code)

    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")
        cards = soup.find_all("div", class_="a-card__descr")

        print("Cards:", len(cards))

        for card in cards:
            title = card.find("a", class_="a-card__title")

            if title == None:
                continue

            link = title["href"].split("?")[0]
            link = link.replace("https://krisha.kz", "")

            if link in seen_links:
                continue

            seen_links.append(link)
            apartments.append(card)

    time.sleep(5)

print("Unique apartments:", len(apartments))

Page: 1 Status: 403
Page: 2 Status: 403
Page: 3 Status: 403


KeyboardInterrupt: 

In [ ]:
data = []

for card in apartments:
    title = card.find("a", class_="a-card__title")
    parts = title.text.split("·")

    rooms = parts[0].split("-")[0].strip()

    area = ""
    floor = ""
    total_floors = ""

    if len(parts) > 1:
        area = parts[1].replace("м²", "").strip()

    if len(parts) > 2:
        floor_text = parts[2].replace("этаж", "").strip()
        floors = floor_text.split("/")
        floor = floors[0].strip()

        if len(floors) > 1:
            total_floors = floors[1].strip()

    price_clean = ""
    price_from = ""

    price = card.find("div", class_="a-card__price")

    if price != None:
        price_original = price.text.strip()
        price_from = price_original.count("от")
        price_text = price_original.replace("₸", "").replace("от", "")
        price_clean = "".join(price_text.split())

    address = ""
    address_element = card.find("div", class_="a-card__subtitle")

    if address_element != None:
        address = address_element.text.strip()

    description = ""
    description_element = card.find("div", class_="a-card__text-preview")

    if description_element != None:
        description = description_element.text.strip()

    link = title["href"].split("?")[0]

    if link.startswith("/"):
        link = "https://krisha.kz" + link

    row = [price_clean, price_from, address, rooms, area,
           floor, total_floors, description, link]

    data.append(row)

print("Rows:", len(data))

Rows: 43


In [ ]:
expanded_data = []

for row in data:
    complex_name = ""
    building_type = ""
    year_built = ""
    ceiling_height = ""

    details = row[7].split(",")

    for detail in details:
        detail = detail.strip()

        if detail.startswith("жил. комплекс "):
            complex_name = detail.replace("жил. комплекс ", "").strip()

        if detail.endswith(" дом"):
            building_type = detail

        if detail.endswith(" г.п."):
            year_built = detail.replace(" г.п.", "").strip()

        if detail.startswith("потолки "):
            ceiling_height = detail.replace("потолки ", "").replace("м.", "").strip()

    new_row = []

    for value in row:
        new_row.append(value)

    new_row.append(complex_name)
    new_row.append(building_type)
    new_row.append(year_built)
    new_row.append(ceiling_height)

    expanded_data.append(new_row)

print("Rows with additional characteristics:", len(expanded_data))

Rows with additional characteristics: 43


In [ ]:
with open("astana_apartments.csv", "w", encoding="utf-8") as file:
    file.write("price,price_from,address,rooms,area,floor,total_floors,description,link,complex_name,building_type,year_built,ceiling_height\n")

    for row in expanded_data:
        values = []

        for value in row:
            text = f"{value}".replace('"', '""')
            values.append('"' + text + '"')

        file.write(",".join(values) + "\n")

print("Saved rows:", len(expanded_data))
print("File: astana_apartments.csv")

Saved rows: 43
File: astana_apartments.csv
